# Poster Figures (v2)

This notebook generates **poster-ready tables/plots** you can screenshot:
1) **Feature set ladder** (F1→F5)
2) **Sharpe by feature set** (Logistic, 3D)

Assumes you already ran:
```bash
python -m src.pipeline --mode live --notebook 02
```
so the CSV outputs exist under `outputs/metrics/`.


In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Make paths robust whether you run from repo root or from notebooks/
HERE = Path.cwd()
# If we are inside notebooks/, go one level up
ROOT = HERE if (HERE / 'outputs').exists() else HERE.parent
OUT_METRICS = ROOT / 'outputs' / 'metrics'
OUT_FIG = ROOT / 'outputs' / 'figures'
OUT_FIG.mkdir(parents=True, exist_ok=True)

print('ROOT =', ROOT)
print('OUT_METRICS =', OUT_METRICS)
print('OUT_FIG =', OUT_FIG)


## 1) Feature set ladder (F1 → F5)

This is the cleanest way to explain the feature-engineering design on the poster.

You can screenshot the rendered table below.

In [ ]:
from src.config import AppConfig

config = AppConfig()
feature_sets = config.feature_sets

rows = []
for fs_name, groups in feature_sets.items():
    rows.append({
        'Feature Set': fs_name,
        'Feature Groups Included': ', '.join(groups),
    })

feature_ladder = pd.DataFrame(rows)
# Order nicely
order = ['F1_momentum','F2_momentum_reversal','F3_plus_risk_liquidity','F4_plus_cross_sectional','F5_full_finance']
feature_ladder['__order'] = feature_ladder['Feature Set'].apply(lambda x: order.index(x) if x in order else 999)
feature_ladder = feature_ladder.sort_values('__order').drop(columns='__order').reset_index(drop=True)

feature_ladder

**Optional (nice for poster):** save the ladder table as an image.
If this works on your machine, you can directly upload the PNG into Canva.

In [ ]:
# Save as PNG (simple matplotlib table)
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
tbl = ax.table(cellText=feature_ladder.values, colLabels=feature_ladder.columns, cellLoc='left', colLoc='left', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.4)
ax.set_title('Feature Set Ladder (F1 → F5)', fontsize=12, pad=12)
out_path = OUT_FIG / 'poster_feature_ladder.png'
plt.tight_layout()
plt.savefig(out_path, dpi=200)
plt.show()
print('Saved:', out_path)

## 2) Load outputs from live run
We will use the existing CSV tables under `outputs/metrics/` to make poster charts.

In [ ]:
metrics_path = OUT_METRICS / '02_live_metrics.csv'
backtest_path = OUT_METRICS / '02_live_backtest_summary.csv'
relative_path = OUT_METRICS / '02_live_relative_summary.csv'

assert metrics_path.exists(), f'Missing: {metrics_path} (run pipeline first)'
assert backtest_path.exists(), f'Missing: {backtest_path} (run pipeline first)'
assert relative_path.exists(), f'Missing: {relative_path} (run pipeline first)'

metrics = pd.read_csv(metrics_path)
backtest = pd.read_csv(backtest_path)
relative = pd.read_csv(relative_path)

print('metrics rows:', len(metrics))
print('backtest rows:', len(backtest))
print('relative rows:', len(relative))
metrics.head(3)

## 3) Chart: Sharpe by feature set (Logistic, 3D)
This directly supports the poster claim: deeper features (F1→F5) improve portfolio outcomes (in our simplified execution-aware backtest).

In [ ]:
# Filter: Logistic + 3D
sub = backtest[(backtest['model'] == 'logistic_regression') & (backtest['horizon_days'] == 3)].copy()
sub = sub.sort_values('feature_set')

# Enforce feature ladder order if available
order = ['F1_momentum','F2_momentum_reversal','F3_plus_risk_liquidity','F4_plus_cross_sectional','F5_full_finance']
sub['__order'] = sub['feature_set'].apply(lambda x: order.index(x) if x in order else 999)
sub = sub.sort_values('__order').drop(columns='__order')

display(sub[['feature_set','sharpe','max_drawdown','annualized_return','annualized_volatility','avg_turnover','avg_cost_drag']])

plt.figure(figsize=(10, 3.8))
plt.bar(sub['feature_set'], sub['sharpe'])
plt.xticks(rotation=20, ha='right')
plt.ylabel('Sharpe')
plt.title('Sharpe by Feature Set (Logistic Regression, 3D)')
plt.tight_layout()
out_path = OUT_FIG / 'poster_sharpe_by_feature_set_logreg_3d.png'
plt.savefig(out_path, dpi=200)
plt.show()
print('Saved:', out_path)

## 4) Optional chart: Balanced accuracy by feature set (Logistic, 3D)
Balanced accuracy differences are small, but this is a clean ML-focused companion plot.

In [ ]:
subm = metrics[(metrics['model'] == 'logistic_regression') & (metrics['horizon_days'] == 3)].copy()
subm['__order'] = subm['feature_set'].apply(lambda x: order.index(x) if x in order else 999)
subm = subm.sort_values('__order').drop(columns='__order')

display(subm[['feature_set','balanced_accuracy','accuracy','rank_ic','top_k_hit_rate','top_bottom_spread']])

plt.figure(figsize=(10, 3.8))
plt.plot(subm['feature_set'], subm['balanced_accuracy'], marker='o', label='balanced_accuracy')
plt.plot(subm['feature_set'], subm['accuracy'], marker='o', label='accuracy', alpha=0.7)
plt.xticks(rotation=20, ha='right')
plt.ylim(0.45, 0.60)
plt.ylabel('Score')
plt.title('Prediction Metrics by Feature Set (Logistic Regression, 3D)')
plt.legend()
plt.tight_layout()
out_path = OUT_FIG / 'poster_pred_metrics_by_feature_set_logreg_3d.png'
plt.savefig(out_path, dpi=200)
plt.show()
print('Saved:', out_path)

## What to screenshot for poster
Recommended screenshots/exports:
- `poster_feature_ladder.png` (or the table output)
- `poster_sharpe_by_feature_set_logreg_3d.png`
- existing `outputs/figures/02_live_rankic_heatmap.png`
- existing `outputs/figures/02_live_F5_full_finance_logistic_regression_3d_equity.png`
